
# Main Report Tables

Generate compact LaTeX tables for the ICML 2026 paper from the cached notebook handoff artifacts.

Outputs:
- `icml2026/tables/nominal_significant_associations_appendix.tex`: one appendix `longtable` with every general-matrix association satisfying raw `p < 0.05`, for every comparison.
- `icml2026/tables/robust_associations.tex`: compact ICML-style `table` with all robust associations (`BH q < 0.05` and bootstrap sign consistency >= 0.90).

The appendix table uses `longtable`, so the paper preamble needs `\usepackage{longtable}` in addition to the already present `booktabs` package.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

In [2]:
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    package_dir = cwd.parent
    repo_dir = package_dir.parent
elif cwd.name == "meta-feature-analysis":
    package_dir = cwd
    repo_dir = cwd.parent
else:
    repo_dir = cwd
    package_dir = repo_dir / "meta-feature-analysis"

handoff_root = package_dir / ".mfa_cache" / "notebook_handoff"
table_dir = repo_dir / "icml2026" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)

# Main comparisons first, then the additional appendix comparisons.
CONFIGS = [
    {
        "key": "config_1",
        "config_dir": "config_1_c4472762a9293af4",
        "comparison": "Non-TFM vs. TFM",
        "short_comparison": "Non-TFM / TFM",
    },
    {
        "key": "config_2",
        "config_dir": "config_2_781b7f9c27428458",
        "comparison": "TabICL v2 vs. TabPFN v2.6",
        "short_comparison": "TabICL v2 / TabPFN v2.6",
    },
    {
        "key": "config_0",
        "config_dir": "config_0_f025f9bfe1d83684",
        "comparison": "NN vs. Tree",
        "short_comparison": "NN / Tree",
    },
    {
        "key": "config_3",
        "config_dir": "config_3_74649dc702cd1b53",
        "comparison": "RealTabPFN v2.5 vs. TabPFN v2.6",
        "short_comparison": "RealTabPFN / TabPFN v2.6",
    },
    {
        "key": "config_4",
        "config_dir": "config_4_b1fcffe10c27ec86",
        "comparison": "TabPFN old vs. TabPFN v2.6",
        "short_comparison": "TabPFN old / new",
    },
    {
        "key": "config_5",
        "config_dir": "config_5_b2edac3115580fa3",
        "comparison": "TabICL old vs. TabICL v2",
        "short_comparison": "TabICL old / new",
    },
]

ASSOCIATION_TABLES = {"general_associations": "General"}
ROBUST_TABLES = {"robust_general_associations": "General"}

P_VALUE_THRESHOLD = 0.05
TOP_K_NOMINAL_FEATURES = 10

In [3]:
def load_handoff_table(config_dir: str, table_name: str) -> pd.DataFrame:
    path = handoff_root / config_dir / f"{table_name}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"Missing handoff artifact: {path}")
    return pd.read_pickle(path).copy()


def load_metadata(config_dir: str) -> dict:
    path = handoff_root / config_dir / "metadata.json"
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


feature_name_map = {
    "log_n": r"$\log n$",
    "log_d": r"$\log d$",
    "d_over_n": r"$d/n$",
    "n_over_d": r"$n/d$",
    "cat_fraction": "categorical fraction",
    "feature_missing_fraction": "feature missing fraction",
    "missing_fraction": "missing fraction",
    "n_classes": "number of classes",
    "class_imbalance_ratio": "class imbalance ratio",
    "effective_rank": "effective rank",
}


def format_feature_name(feature: str) -> str:
    if feature in feature_name_map:
        return feature_name_map[feature]
    if feature.startswith("pymfe__"):
        body = feature.removeprefix("pymfe__")
        if "." in body:
            base, summary = body.rsplit(".", 1)
            return f"MFE {base.replace('_', ' ')} ({summary.replace('_', ' ')})"
        return f"MFE {body.replace('_', ' ')}"
    return feature.replace("_", " ")


def latex_escape(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    # Preserve math labels intentionally returned by format_feature_name.
    if text.startswith("$") and text.endswith("$"):
        return text
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in text)


def format_decimal(value: object, digits: int = 3) -> str:
    if value is None or pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"


def format_p_value(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    value = float(value)
    if value < 0.001:
        return r"$<.001$"
    return f"{value:.3f}".lstrip("0")


def format_ci(low: object, high: object) -> str:
    if pd.isna(low) or pd.isna(high):
        return ""
    return f"[{format_decimal(low)}, {format_decimal(high)}]"


def make_booktabs_table(
    rows: list[list[str]],
    *,
    headers: list[str],
    column_spec: str,
    caption: str,
    label: str,
    table_env: str = "table",
    size: str = "small",
    placement: str = "t",
) -> str:
    row_end = " \\\\"
    header = " & ".join(headers) + row_end
    body = "\n".join("        " + " & ".join(row) + row_end for row in rows)
    if not body:
        body = (
            "        "
            + r"\multicolumn{"
            + str(len(headers))
            + r"}{c}{No associations met the reporting criterion.}"
            + row_end
        )
    return f"""\\begin{{{table_env}}}[{placement}]
  \\caption{{{caption}}}
  \\label{{{label}}}
  \\begin{{center}}
    \\begin{{{size}}}
      \\setlength{{\\tabcolsep}}{{2pt}}
      \\begin{{tabular}}{{{column_spec}}}
        \\toprule
        {header}
        \\midrule
{body}
        \\bottomrule
      \\end{{tabular}}
    \\end{{{size}}}
  \\end{{center}}
  \\vskip -0.1in
\\end{{{table_env}}}
"""

In [4]:
association_frames = []
robust_frames = []

for order, spec in enumerate(CONFIGS):
    metadata = load_metadata(spec["config_dir"])
    config_file = metadata.get("config_file", spec["key"] + ".yaml")
    for table_name, analysis_label in ASSOCIATION_TABLES.items():
        frame = load_handoff_table(spec["config_dir"], table_name)
        frame = frame.loc[frame["p_value"].notna()].copy()
        frame.insert(0, "config_order", order)
        frame.insert(1, "config", spec["key"])
        frame.insert(2, "config_file", config_file)
        frame.insert(3, "comparison", spec["comparison"])
        frame.insert(4, "short_comparison", spec["short_comparison"])
        frame.insert(5, "analysis", analysis_label)
        association_frames.append(frame)
    for table_name, analysis_label in ROBUST_TABLES.items():
        frame = load_handoff_table(spec["config_dir"], table_name).copy()
        frame.insert(0, "config_order", order)
        frame.insert(1, "config", spec["key"])
        frame.insert(2, "config_file", config_file)
        frame.insert(3, "comparison", spec["comparison"])
        frame.insert(4, "short_comparison", spec["short_comparison"])
        frame.insert(5, "analysis", analysis_label)
        robust_frames.append(frame)

associations = pd.concat(association_frames, ignore_index=True)
robust_associations = pd.concat(robust_frames, ignore_index=True)

nominal_significant = (
    associations.loc[associations["p_value"].lt(P_VALUE_THRESHOLD)]
    .sort_values(
        ["config_order", "p_value", "abs_spearman_r", "feature"],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)
robust_associations = (
    robust_associations
    .sort_values(
        ["config_order", "abs_spearman_rho", "feature"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

summary = pd.DataFrame(
    {
        "comparison": spec["comparison"],
        "nominal_p_lt_0_05": int(
            nominal_significant.loc[
                nominal_significant["config"].eq(spec["key"])
            ].shape[0]
        ),
        "robust_associations": int(
            robust_associations.loc[
                robust_associations["config"].eq(spec["key"])
            ].shape[0]
        ),
    }
    for spec in CONFIGS
)
summary

,comparison,nominal_p_lt_0_05,robust_associations
0,Non-TFM vs. TFM,31,1
1,TabICL v2 vs. TabPFN v2.6,34,1
2,NN vs. Tree,11,0
3,RealTabPFN v2.5 vs. TabPFN v2.6,15,0
4,TabPFN old vs. TabPFN v2.6,8,0
5,TabICL old vs. TabICL v2,16,0


In [5]:
def comparison_slug(config_key: str) -> str:
    return config_key.replace("_", "-")


def nominal_table_for_comparison(rows_frame: pd.DataFrame, spec: dict) -> str:
    headers = ["Feature", "$n$", r"$\rho$", "$p$", "$q_{BH}$"]
    rows = []
    for row in rows_frame.itertuples(index=False):
        rows.append(
            [
                latex_escape(format_feature_name(row.feature)),
                str(int(row.n)) if not pd.isna(row.n) else "",
                format_decimal(row.spearman_r),
                format_p_value(row.p_value),
                format_p_value(row.p_value_bh),
            ]
        )
    return make_booktabs_table(
        rows,
        headers=headers,
        column_spec=r"@{}p{0.48\textwidth}rrrr@{}",
        caption=(
            f"Top nominally significant general-matrix meta-feature associations for {spec['comparison']}. "
            rf"Rows show up to {TOP_K_NOMINAL_FEATURES} features with raw Spearman $p<0.05$; "
            r"$q_{BH}$ is reported for context."
        ),
        label=f"tab:appendix_nominal_{comparison_slug(spec['key'])}",
        table_env="table*",
        size="scriptsize",
        placement="t",
    )


nominal_paths = []
nominal_tables = {}
for spec in CONFIGS:
    table_frame = nominal_significant.loc[nominal_significant["config"].eq(spec["key"])].head(TOP_K_NOMINAL_FEATURES).copy()
    latex = nominal_table_for_comparison(table_frame, spec)
    path = table_dir / f"nominal_significant_associations_{comparison_slug(spec['key'])}.tex"
    path.write_text(latex, encoding="utf-8")
    nominal_paths.append(path)
    nominal_tables[spec["key"]] = latex

nominal_index_path = table_dir / "nominal_significant_associations_appendix.tex"
nominal_index_latex = "\n".join(f"\\input{{tables/{path.name}}}" for path in nominal_paths) + "\n"
nominal_index_path.write_text(nominal_index_latex, encoding="utf-8")

pd.DataFrame({"nominal_table": [str(path) for path in nominal_paths + [nominal_index_path]]})


,nominal_table
0,/work/mherre/tabular-meta-feature-analysis/icm...
1,/work/mherre/tabular-meta-feature-analysis/icm...
2,/work/mherre/tabular-meta-feature-analysis/icm...
3,/work/mherre/tabular-meta-feature-analysis/icm...
4,/work/mherre/tabular-meta-feature-analysis/icm...
5,/work/mherre/tabular-meta-feature-analysis/icm...
6,/work/mherre/tabular-meta-feature-analysis/icm...


In [6]:
def robust_table(rows_frame: pd.DataFrame) -> str:
    headers = ["Comparison", "Feature", "$n$", r"$\rho$ [95\% CI]", "$q_{BH}$", "Sign", "Top-25"]
    rows = []
    for row in rows_frame.itertuples(index=False):
        rows.append(
            [
                latex_escape(row.short_comparison),
                latex_escape(format_feature_name(row.feature)),
                str(int(row.n)) if not pd.isna(row.n) else "",
                latex_escape(
                    f"{format_decimal(row.spearman_rho)} {format_ci(row.bootstrap_ci_low, row.bootstrap_ci_high)}"
                ),
                format_p_value(row.bh_q_value),
                format_decimal(row.bootstrap_sign_consistency, 2),
                format_decimal(row.bootstrap_top_25_frequency, 2),
            ]
        )
    return make_booktabs_table(
        rows,
        headers=headers,
        column_spec=r"@{}p{0.20\textwidth}p{0.30\textwidth}rrrrr@{}",
        caption=(
            "Robust meta-feature associations. Robust rows pass BH $q<0.05$ and bootstrap sign consistency at least 0.90."
        ),
        label="tab:robust_associations",
        table_env="table*",
        size="scriptsize",
        placement="t",
    )


robust_latex = robust_table(robust_associations)
robust_path = table_dir / "robust_associations.tex"
robust_path.write_text(robust_latex, encoding="utf-8")
print(robust_path)

/work/mherre/tabular-meta-feature-analysis/icml2026/tables/robust_associations.tex


In [7]:
display(Markdown("### Nominal significant associations preview"))
display(
    nominal_significant.assign(
        feature_label=lambda df: df["feature"].map(format_feature_name)
    )[
        [
            "comparison",
            "feature_label",
            "n",
            "spearman_r",
            "p_value",
            "p_value_bh",
        ]
    ].head(
        20
    )
)

display(Markdown("### Robust associations preview"))
display(
    robust_associations.assign(
        feature_label=lambda df: df["feature"].map(format_feature_name)
    )[
        [
            "comparison",
            "feature_label",
            "n",
            "spearman_rho",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "bh_q_value",
            "bootstrap_sign_consistency",
            "bootstrap_top_25_frequency",
        ]
    ]
)

### Nominal significant associations preview

,comparison,feature_label,n,spearman_r,p_value,p_value_bh
0,Non-TFM vs. TFM,MFE attr ent (skewness),50,-0.510204,0.000154,0.043331
1,Non-TFM vs. TFM,MFE attr ent.histogram (9),50,0.404514,0.003572,0.309243
2,Non-TFM vs. TFM,MFE cor.histogram (6),43,0.427391,0.004256,0.309243
3,Non-TFM vs. TFM,MFE sparsity (skewness),44,0.421424,0.004386,0.309243
4,Non-TFM vs. TFM,MFE sparsity.histogram (0),45,0.395309,0.007196,0.346616
5,Non-TFM vs. TFM,MFE attr conc (median),50,0.374502,0.007375,0.346616
6,Non-TFM vs. TFM,MFE sparsity (median),45,-0.364954,0.013702,0.359709
7,Non-TFM vs. TFM,MFE kurtosis.histogram (4),43,0.369898,0.014631,0.359709
8,Non-TFM vs. TFM,MFE sparsity.quantiles (1),45,-0.361660,0.014643,0.359709
9,Non-TFM vs. TFM,MFE cor.histogram (8),43,0.369357,0.014787,0.359709


### Robust associations preview

,comparison,feature_label,n,spearman_rho,bootstrap_ci_low,bootstrap_ci_high,bh_q_value,bootstrap_sign_consistency,bootstrap_top_25_frequency
0,Non-TFM vs. TFM,MFE attr ent (skewness),50,-0.510204,-0.709497,-0.230523,0.043331,1.0,0.914
1,TabICL v2 vs. TabPFN v2.6,MFE attr conc (median),50,-0.510108,-0.725038,-0.258277,0.043476,1.0,0.874


In [8]:
display(Markdown("### LaTeX snippets"))
first_key = CONFIGS[0]["key"]
print(f"Nominal table ({CONFIGS[0]['comparison']}):")
print(nominal_tables[first_key])
print("\nNominal appendix include file:")
print(nominal_index_latex)
print("\nRobust table:")
print(robust_latex)


### LaTeX snippets

Nominal table (Non-TFM vs. TFM):
\begin{table*}[t]
  \caption{Top nominally significant general-matrix meta-feature associations for Non-TFM vs. TFM. Rows show up to 10 features with raw Spearman $p<0.05$; $q_{BH}$ is reported for context.}
  \label{tab:appendix_nominal_config-1}
  \begin{center}
    \begin{scriptsize}
      \setlength{\tabcolsep}{2pt}
      \begin{tabular}{@{}p{0.48\textwidth}rrrr@{}}
        \toprule
        Feature & $n$ & $\rho$ & $p$ & $q_{BH}$ \\
        \midrule
        MFE attr ent (skewness) & 50 & -0.510 & $<.001$ & .043 \\
        MFE attr ent.histogram (9) & 50 & 0.405 & .004 & .309 \\
        MFE cor.histogram (6) & 43 & 0.427 & .004 & .309 \\
        MFE sparsity (skewness) & 44 & 0.421 & .004 & .309 \\
        MFE sparsity.histogram (0) & 45 & 0.395 & .007 & .347 \\
        MFE attr conc (median) & 50 & 0.375 & .007 & .347 \\
        MFE sparsity (median) & 45 & -0.365 & .014 & .360 \\
        MFE kurtosis.histogram (4) & 43 & 0.370 & .015 & .360 \\
    